# Análise de Medalhas\n\nQuadro de medalhas por país ao longo de todas as edições olímpicas.

In [1]:
import pandas as pd
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

In [2]:
BRONZE_DIR = Path("../../bronze")
OUTPUT_DIR = Path(".")

df_medalhas = pd.read_parquet(BRONZE_DIR / "medalhas_1986_2024.parquet")

quadro = df_medalhas.pivot_table(
    index="pais",
    columns="medalha",
    values="atleta",
    aggfunc="count",
    fill_value=0
).reset_index()

for col in ["Gold", "Silver", "Bronze"]:
    if col not in quadro.columns:
        quadro[col] = 0

quadro = quadro[["pais", "Gold", "Silver", "Bronze"]]
quadro["Total"] = quadro[["Gold", "Silver", "Bronze"]].sum(axis=1)
quadro.columns.name = None
quadro = quadro.sort_values(["Gold", "Silver", "Bronze"], ascending=False).reset_index(drop=True)

quadro.to_csv(OUTPUT_DIR / "medalhas_summary.csv", index=False)

print(f"Quadro de medalhas: {len(quadro)} paises")
quadro.head(15)

Quadro de medalhas: 160 paises


,pais,Gold,Silver,Bronze,Total
0,USA,3076,1956,1592,6624
1,URS,1099,743,704,2546
2,GBR,852,836,846,2534
3,GER,844,832,881,2557
4,ITA,628,600,611,1839
5,FRA,622,782,759,2163
6,CAN,596,545,591,1732
7,SWE,529,586,557,1672
8,CHN,495,486,385,1366
9,HUN,466,356,427,1249


In [3]:
top15 = quadro.head(15).set_index("pais")

fig, ax = plt.subplots(figsize=(12, 6))
top15[["Gold", "Silver", "Bronze"]].plot(
    kind="barh",
    stacked=True,
    color=["#FFD700", "#C0C0C0", "#CD7F32"],
    ax=ax
)
ax.set_xlabel("Quantidade de Medalhas")
ax.set_ylabel("")
ax.set_title("Top 15 Países - Quadro de Medalhas")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "medalhas_plot.png", dpi=150, bbox_inches="tight")
plt.show()

In [4]:
metadata = {
    "nome": "Análise de Medalhas",
    "fonte": "Camada bronze - medalhas_1986_2024",
    "descricao": "Quadro de medalhas por país com visualização dos top 15",
    "formato": "csv/png",
    "colunas": [{"nome": col, "tipo": str(quadro[col].dtype)} for col in quadro.columns],
    "quantidade_linhas": len(quadro),
    "data_coleta": datetime.now().strftime("%Y-%m-%d"),
    "observacoes": "Contagem individual de medalhas por atleta"
}

with open(OUTPUT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=4)